In [1]:
import pandas as pd
import vllm
import torch

/home/li.mil/miniconda3/envs/elemental_tasks/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/li.mil/miniconda3/envs/elemental_tasks/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


INFO 07-07 18:00:05 [__init__.py:243] Automatically detected platform cuda.


2025-07-07 18:00:07,835	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# The question: *When* does a model learn to acknowledge ICL?
# First, need to run through to evaluate whether models can do a specific task

# These settings are all under ICL...
# Algorithmic: Uppercase a letter (a -> A), lowercase a letter (A -> a), list the first letter, list the last letter
# Translation: Eng -> Fr, Fr -> Eng; Eng -> Sp, Sp -> Eng
# Linguistic: Present to gerund, singular to plural; Antonyms and synonyms have already been done
# Knowledge (factual): Country -> Capital, Country -> Currency

task_categories_to_examples = {
    "uppercase": ["a -> A", "c -> C"],
    "lowercase": ["A -> a", "C -> c"],
    "first_letter": ["the cat went up the tree -> t", "elephants are cool -> e"],
    "last_letter": ["the cat went up the tree -> e", "elephants are cool -> l"],
    "translate_eng_fr": ["hello -> bonjour", "goodbye -> au revoir"],
    "translate_fr_eng": ["bonjour -> hello", "au revoir -> goodbye"],
    "translate_eng_sp": ["hello -> hola", "goodbye -> adiós"],
    "translate_sp_eng": ["hola -> hello", "adiós -> goodbye"],
    "present_to_gerund": ["run -> running", "swim -> swimming"],
    "singular_to_plural": ["cat -> cats", "dog -> dogs"],
    "country_to_capital": ["France -> Paris", "Germany -> Berlin"],
    "country_to_currency": ["France -> Euro", "United States -> Dollar"],
}

def craft_icl(category):
    examples = task_categories_to_examples[category]
    prompt = ""
    for example in examples:
        prompt += f"{example}\n"
    return prompt

# Load the simple csv with pandas
data = pd.read_csv("/home/li.mil/ElementalTask/dataset/simple.csv")
data = data.drop(columns=["index"])

In [3]:
all_prompts = []
all_answers = []
for idx, row in data.iterrows():
    category = row['category_name']
    icl_prompt = craft_icl(category)
    icl_prompt += f"{row['question']} ->"
    all_prompts.append(icl_prompt)
    all_answers.append(row['answer'])

In [4]:
# Generate and eval
model = vllm.LLM(
    model="allenai/OLMo-2-1124-7B",
    tokenizer="allenai/OLMo-2-1124-7B",
    revision=None,
    tokenizer_mode="auto",
    tensor_parallel_size=torch.cuda.device_count(),
    trust_remote_code=True,
)

sampling_params = vllm.SamplingParams(
    temperature=0,
    max_tokens=10,
)

outputs = model.generate(all_prompts, sampling_params)
outputs = [it.outputs[0].text for it in outputs]

INFO 07-07 18:00:07 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 07-07 18:00:07 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 07-07 18:00:07 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 07-07 18:00:08 [config.py:3131] Downcasting torch.float32 to torch.float16.
INFO 07-07 18:00:17 [config.py:793] This model supports multiple tasks: {'classify', 'embed', 'score', 'generate', 'reward'}. Defaulting to 'generate'.
INFO 07-07 18:00:17 [config.py:2118] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-07 18:00:18 [core.py:438] Waiting for init message from front-end.
INFO 07-07 18:00:18 [core.py:65] Initializing a V1 LLM engine (v0.9.0.1) with config: model='allenai/OLMo-2-1124-7B', speculative_config=None, tokenizer='allenai/OLMo-2-1124-7B', skip_tokenizer_init=False, tokenizer_mode=auto, revi

Loading safetensors checkpoint shards:   0% Completed | 0/6 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  17% Completed | 1/6 [00:00<00:02,  2.50it/s]
Loading safetensors checkpoint shards:  33% Completed | 2/6 [00:00<00:01,  2.32it/s]
Loading safetensors checkpoint shards:  50% Completed | 3/6 [00:01<00:01,  2.29it/s]
Loading safetensors checkpoint shards:  67% Completed | 4/6 [00:01<00:00,  2.30it/s]
Loading safetensors checkpoint shards:  83% Completed | 5/6 [00:02<00:00,  2.31it/s]
Loading safetensors checkpoint shards: 100% Completed | 6/6 [00:02<00:00,  2.31it/s]
Loading safetensors checkpoint shards: 100% Completed | 6/6 [00:02<00:00,  2.31it/s]



INFO 07-07 18:00:22 [default_loader.py:280] Loading weights took 2.65 seconds
INFO 07-07 18:00:23 [gpu_model_runner.py:1549] Model loading took 13.5958 GiB and 3.043201 seconds
INFO 07-07 18:00:24 [kv_cache_utils.py:637] GPU KV cache size: 114,880 tokens
INFO 07-07 18:00:24 [kv_cache_utils.py:640] Maximum concurrency for 4,096 tokens per request: 28.05x
INFO 07-07 18:00:28 [gpu_model_runner.py:1933] Graph capturing finished in 4 secs, took 0.14 GiB
INFO 07-07 18:00:28 [core.py:167] init engine (profile, create kv cache, warmup model) took 5.46 seconds


Processed prompts: 100%|██████████| 116/116 [00:00<00:00, 314.24it/s, est. speed input: 4267.60 toks/s, output: 3143.08 toks/s]


In [23]:
cat_to_score = {}
for cat, out, ans in zip(data.iterrows(), outputs, all_answers):
    category = cat[1]['category_name']
    if category not in cat_to_score:
        cat_to_score[category] = []
    if out.split("\n")[0].strip() == ans.strip():
        cat_to_score[category].append(1)
    else:
        cat_to_score[category].append(0)

    # print(f"Output: {out.split("\n")[0].strip()}\nExpected: {ans.strip()}\nMatch: {out.split("\n")[0].strip() == ans.strip()}\n")

# Calculate the scores
for category, scores in cat_to_score.items():
    score = sum(scores) / len(scores) if scores else 0
    print(f"Category: {category}, Score: {score:.2f}")

Category: uppercase, Score: 1.00
Category: lowercase, Score: 1.00
Category: first_letter, Score: 0.80
Category: last_letter, Score: 0.10
Category: translate_eng_fr, Score: 0.80
Category: translate_fr_eng, Score: 1.00
Category: translate_eng_sp, Score: 0.90
Category: translate_sp_eng, Score: 1.00
Category: present_to_gerund, Score: 0.90
Category: singular_to_plural, Score: 1.00
Category: country_to_capital, Score: 0.60
Category: country_to_currency, Score: 0.60
